In [2]:
import os
import glob
import polars as pl
from pathlib import Path

# ==========================================
# 1. CONFIGURATION & PATHS
# ==========================================
# Feature Data Directory (Where the CSVs to be updated are located)

# Change the Drive letter to "F" or "D" depending on the mounted SSD drive in the PC

#======================================================================================================================
# Select this path  for training data updates
#FEAT_DIR = Path(r"D:\Research_Data_Public\Esports\CSDATA\DataScience\CS2_Project\FeatData\Wfire_STOMS_01\Training\Ancient")

# Select this path  for testing data
FEAT_DIR = Path(r"D:\Research_Data_Public\Esports\CSDATA\DataScience\CS2_Project\FeatData\Wfire_STOMS_01\Testing\Ancient")

# Raw Data Directory (To extract the 'round_end' ground truth)
RAW_DATA_DIR = Path(r"D:\Research_Data_Public\Esports\CSDATA\datasets\billpureskillgg\cs2-2023-11-23\versions\2\csds")

# Standardized Team Mapping
TEAM_MAP = {"2": "T", "3": "CT", "TERRORIST": "T", "CT": "CT", "T": "T"}

# ==========================================
# 2. DATA RETRIEVAL & UPDATING LOGIC
# ==========================================
def update_features_with_targets():
    """Iterates through feature CSVs, retrieves ground-truth labels, and appends them."""
    
    if not FEAT_DIR.exists():
        print(f"Error: Feature directory does not exist -> {FEAT_DIR}")
        return

    csv_files = list(FEAT_DIR.glob("*.csv"))
    if not csv_files:
        print(f"No CSV files found in -> {FEAT_DIR}")
        return

    print(f"Found {len(csv_files)} matches to update. Processing...\n")
    updated_count = 0

    for filepath in csv_files:
        match_id = filepath.stem
        
        # --- A. Find the Raw Round_End Data ---
        match_dirs = list(RAW_DATA_DIR.rglob(match_id))
        if not match_dirs:
            print(f"Skipping {match_id}: Raw data directory not found.")
            continue
            
        re_files = glob.glob(str(match_dirs[0] / "round_end*"))
        if not re_files:
            print(f"Skipping {match_id}: 'round_end' channel missing.")
            continue

        # --- B. Extract the Target Labels ---
        try:
            df_re = pl.read_parquet(re_files[0])
            
            # Map the winner codes and keep only 'round' and 'target'
            df_labels = df_re.with_columns(
                pl.col("winner_team_code").cast(pl.Utf8).replace(TEAM_MAP).alias("target")
            ).select(["round", "target"])
            
        except Exception as e:
            print(f"Error reading 'round_end' for {match_id}: {e}")
            continue

        # --- C. Update the Feature Data ---
        try:
            df_feat = pl.read_csv(filepath)
            
            # Safety check: If "target" already exists from a previous run, drop it to prevent duplicates like "target_right"
            if "target" in df_feat.columns:
                df_feat = df_feat.drop("target")
                
            # Perform a Left Join on the "round" column
            # This ensures every round in the feature data gets its corresponding label
            df_updated = df_feat.join(df_labels, on="round", how="left")
            
            # --- D. Save the Changes (Overwrite) ---
            df_updated.write_csv(filepath)
            updated_count += 1
            
            print(f"Successfully updated: {match_id}")
            
        except Exception as e:
            print(f"Error updating feature file {match_id}: {e}")

    print("\n" + "="*40)
    print(f" UPDATE COMPLETE: {updated_count}/{len(csv_files)} files modified.")
    print("="*40)

# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    # Note: Because this script overwrites data, ensure no other processes are locking the CSV files.
    update_features_with_targets()

Found 23 matches to update. Processing...

Successfully updated: ed21c164-6a69-429c-bbf5-27eb555decd7
Successfully updated: 72cd9753-f003-4f0e-b229-fbbe992d8c19
Successfully updated: 5de6195c-5b14-4234-8670-ab5e72486a02
Successfully updated: 8522d89d-3f5a-45de-94c2-598929025561
Successfully updated: 098c1484-25ce-484a-a3bb-a884646abffd
Successfully updated: 6fa49016-4ef3-4ebd-beca-673845b0923f
Successfully updated: 64798b30-c141-4097-9b36-95d87ab70704
Successfully updated: a075c294-30bc-43d2-b0bb-3154dd2d2273
Successfully updated: ce84dd39-b58e-4933-9694-5056403f1918
Successfully updated: a84d0f8d-89cd-4658-80c2-8c0cff83a713
Successfully updated: f3dfbfab-5884-4f15-8f60-db39eda22049
Successfully updated: d9b306a1-dc1c-4957-818a-b4322fd522df
Successfully updated: bafe006b-13e0-45e3-9a84-d827772afd8f
Successfully updated: 560748c7-c75b-447a-8799-ea8e5172c857
Successfully updated: ac9656d5-46be-408c-83a2-7bf1130006ea
Successfully updated: 33f4e3ac-11af-41df-9997-448b8dd96a70
Successfully 